# 扩散模型基础理论

本笔记本将介绍扩散模型(Diffusion Models)的基础理论知识。我们将从以下几个方面来学习：

1. 什么是扩散模型？
2. 扩散过程(Forward Process)
3. 反向过程(Reverse Process)
4. 损失函数设计
5. 与其他生成模型的比较

## 1. 什么是扩散模型？

扩散模型是一类生成模型，其核心思想是通过逐步向数据添加噪声（前向过程），然后学习如何逐步去除噪声（反向过程）来生成数据。这个过程类似于物理中的扩散现象，因此得名。

在本项目中，扩散模型被用于生成车辆的轨迹，这是一个连续的序列生成任务。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# 设置随机种子以保证结果可复现
np.random.seed(42)
torch.manual_seed(42)

def plot_diffusion_process(x0, num_steps=5):
    """
    可视化扩散过程
    x0: 初始数据
    num_steps: 扩散步数
    """
    betas = np.linspace(0.1, 0.9, num_steps)  # beta schedule
    alphas = 1 - betas
    alpha_bars = np.cumprod(alphas)
    
    plt.figure(figsize=(15, 3))
    for t in range(num_steps):
        plt.subplot(1, num_steps, t+1)
        # 计算t时刻的数据
        noise = np.random.randn(*x0.shape)
        xt = np.sqrt(alpha_bars[t]) * x0 + np.sqrt(1 - alpha_bars[t]) * noise
        plt.plot(xt)
        plt.title(f't={t}')
    plt.tight_layout()
    plt.show()


## 2. 扩散过程(Forward Process)

扩散过程是一个逐步向数据添加高斯噪声的过程。在每一步t，我们都会向数据添加一定量的噪声，这个量由超参数β（beta schedule）控制。

数学表达式：
```
q(x_t|x_{t-1}) = N(x_t; \sqrt{1-β_t}x_{t-1}, β_tI)
```

其中：
- x_t 是t时刻的数据
- β_t 是t时刻的噪声系数
- N 表示正态分布

让我们通过一个简单的例子来可视化这个过程：


In [ ]:
# 创建一个简单的正弦波作为原始数据
t = np.linspace(0, 2*np.pi, 100)
x0 = np.sin(t)

# 可视化扩散过程
plot_diffusion_process(x0)
